# Super-resolution-driven decomposition of mixed-class agricultural polygons

Research question: does single-image super-resolution of Sentinel-2 (10 m → 2.5 m) make the component crops of LDD mixed-class parcels (e.g. `A403/A419` = Durian + Mangosteen) spectrally separable by unsupervised clustering? Each mixed pixel at 10 m is, by hypothesis, a sub-pixel mixture of two crops; super-resolution should partially un-mix it.

The experiment compares K=2 KMeans on pure-component and mixed-component pixels at both grids — same area of interest, same LDD shapefile, same feature schema — and reports per-pair purity, split ratio, and a UMAP embedding.

Workflow:

1. Catalogue minority-only mixed-component LU_CODEs from the LDD shapefile.
2. Fetch Sentinel-2 L2A monthly medians via CDSE openEO (10 m LR) and load the SR (2.5 m) cache.
3. Rasterise labels and sample per-pixel features on both grids.
4. Cluster pure-minority pixels and inspect intrinsic separability.
5. Per-pair K=2 KMeans on (pure A + pure B + mixed A/B), LR vs SR side-by-side.
6. Export the decomposed mixed-pixel pool.


In [1]:
from __future__ import annotations
import sys, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import rasterio
import rioxarray as rxr
import xarray as xr
import geopandas as gpd
from rasterio.features import rasterize

warnings.filterwarnings("ignore", category=UserWarning)
print("python:", sys.version.split()[0])


python: 3.12.13


In [2]:
QUADRANTS         = ["nw", "ne", "sw", "se"]
LR_MONTHS         = ["2024-10", "2024-11", "2024-12"]
STRIDE_SR         = 2
STRIDE_LR         = 1
MAX_PIX_PER_CLASS = 1_000_000

MINORITY_TAXONOMY = {
    "A403": "Durian", "A404": "Rambutan", "A405": "Coconut", "A407": "Mango",
    "A413": "Longan", "A416": "Jackfruit", "A419": "Mangosteen", "A420": "Langsat",
}

_W, _S, _E, _N = 100.9845, 12.5834, 101.8305, 13.1635
_CLNG, _CLAT   = 101.4291, 12.8539
QUADRANT_BBOX = {
    "nw": (_W,    _CLAT, _CLNG, _N),
    "ne": (_CLNG, _CLAT, _E,    _N),
    "sw": (_W,    _S,    _CLNG, _CLAT),
    "se": (_CLNG, _S,    _E,    _CLAT),
}

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT  = REPO_ROOT / "data"
CACHE_ROOT = DATA_ROOT / "_cache"
SHP_DIR    = DATA_ROOT / "landuse_ryg"
OUT_DIR    = DATA_ROOT / "_out" / "cluster_minority"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def sr_dir_for(q: str) -> Path:
    return CACHE_ROOT / "s2_sr" / f"rayong_{q.lower()}"

def lr_dir_for(q: str) -> Path:
    return CACHE_ROOT / "s2_monthly" / f"rayong_{q.lower()}"

AOI_TAG = "-".join(q.lower() for q in QUADRANTS)
print(f"quadrants {QUADRANTS}  tag={AOI_TAG}")
print(f"months    {LR_MONTHS}")
print(f"minority  {list(MINORITY_TAXONOMY.values())}")


quadrants ['nw', 'ne', 'sw', 'se']  tag=nw-ne-sw-se
months    ['2024-10', '2024-11', '2024-12']
minority  ['Durian', 'Rambutan', 'Coconut', 'Mango', 'Longan', 'Jackfruit', 'Mangosteen', 'Langsat']


## 1 · Shapefile · mixed-component catalogue

LDD encodes mixed parcels with slash-separated codes. Keep only minority-only mixed codes (every component in `MINORITY_TAXONOMY`).


In [3]:
shp_files = list(SHP_DIR.rglob("*.shp"))
if not shp_files:
    raise FileNotFoundError(f"no .shp under {SHP_DIR}")
LU_SHP = shp_files[0]
lu_all = gpd.read_file(LU_SHP)
print(f"shapefile: {LU_SHP.name} · rows={len(lu_all)} · crs={lu_all.crs}")

shapefile: LU_RYG_2567.shp · rows=65136 · crs=EPSG:32647


In [4]:
MINORITY_CODES = set(MINORITY_TAXONOMY)

def _components(code: str) -> list[str]:
    if not isinstance(code, str):
        return []
    return [c.strip() for c in code.split("/") if c.strip()]

lu_all["_components"]   = lu_all["LU_CODE"].apply(_components)
lu_all["_n_components"] = lu_all["_components"].apply(len)
lu_all["_all_minority"] = lu_all["_components"].apply(
    lambda comps: len(comps) > 0 and all(c in MINORITY_CODES for c in comps)
)

pure_minority  = lu_all[(lu_all["_n_components"] == 1) & lu_all["_all_minority"]]
mixed_minority = lu_all[(lu_all["_n_components"] >  1) & lu_all["_all_minority"]]
print(f"pure minority polygons:  {len(pure_minority)}")
print(f"mixed minority polygons: {len(mixed_minority)}\n")

vc_pure  = pure_minority ["LU_CODE"].value_counts()
vc_mixed = mixed_minority["LU_CODE"].value_counts()

print("pure minority by LU_CODE:")
for code, n in vc_pure.items():
    print(f"  {code:<8s} {MINORITY_TAXONOMY[code]:<12s} {n:>5d}")

print("\nmixed minority combinations (top 20):")
for code, n in vc_mixed.head(20).items():
    names = [MINORITY_TAXONOMY[c] for c in _components(code)]
    print(f"  {code:<12s} {' + '.join(names):<32s} {n:>4d}")


pure minority polygons:  7712
mixed minority polygons: 1155

pure minority by LU_CODE:
  A403     Durian        5585
  A405     Coconut        508
  A416     Jackfruit      448
  A419     Mangosteen     418
  A407     Mango          390
  A404     Rambutan       225
  A413     Longan         104
  A420     Langsat         34

mixed minority combinations (top 20):
  A403/A419    Durian + Mangosteen               664
  A403/A416    Durian + Jackfruit                137
  A403/A404    Durian + Rambutan                  87
  A404/A419    Rambutan + Mangosteen              62
  A419/A420    Mangosteen + Langsat               55
  A403/A420    Durian + Langsat                   47
  A416/A419    Jackfruit + Mangosteen             35
  A405/A407    Coconut + Mango                    12
  A413/A419    Longan + Mangosteen                11
  A403/A413    Durian + Longan                    11
  A413/A416    Longan + Jackfruit                  6
  A403/A407    Durian + Mango                      

## 2 · Sentinel-2 rasters · LR (10 m) and SR (2.5 m)

LR pulled from CDSE openEO into `data/_cache/s2_monthly/rayong_<q>/`. SR (super-resolved) already cached under `data/_cache/s2_sr/rayong_<q>/`. Three months: 2024-10 / 11 / 12.


In [ ]:
# Download Sentinel-2 L2A monthly medians for every quadrant that does not yet
# have its `data/_cache/s2_monthly/rayong_<q>/openEO_*.tif` cache.

S2_TARGET_CRS = "EPSG:32647"
S2_EDGE_BUFFER_DEG = 0.005
LR_BANDS = ["B02", "B03", "B04", "B08"]
SCL_BAND = "SCL"

def _need_fetch(q: str) -> bool:
    d = lr_dir_for(q)
    if not d.exists():
        return True
    have = {p.name for p in d.glob("openEO_*.tif")}
    needed = {f"openEO_{m}-01Z.tif" for m in LR_MONTHS}
    return not needed.issubset(have)

_pending = [q for q in QUADRANTS if _need_fetch(q)]
if not _pending:
    print("LR cache complete for every quadrant — skipping openEO fetch.")
else:
    print(f"LR cache missing for: {_pending}")
    import openeo
    CONN = openeo.connect("openeo.dataspace.copernicus.eu")
    try:
        CONN.authenticate_oidc()
    except Exception as e:
        print("auth needed:", e)
    print("connected:", CONN.capabilities().get("title", "CDSE openEO"))

    _start = f"{LR_MONTHS[0]}-01"
    _end_month = pd.to_datetime(f"{LR_MONTHS[-1]}-01") + pd.offsets.MonthEnd(0)
    _end = _end_month.strftime("%Y-%m-%d")

    for q in _pending:
        w, s, e, n = QUADRANT_BBOX[q]
        b = S2_EDGE_BUFFER_DEG
        bbox = dict(west=w - b, south=s - b, east=e + b, north=n + b, crs="EPSG:4326")
        out = lr_dir_for(q); out.mkdir(parents=True, exist_ok=True)
        print(f"\n[{q.upper()}] bbox={bbox} months={LR_MONTHS} -> {out}")

        cube = CONN.load_collection(
            "SENTINEL2_L2A",
            spatial_extent=bbox,
            temporal_extent=[_start, _end],
            bands=LR_BANDS + [SCL_BAND],
            max_cloud_cover=85,
        )
        scl = cube.band(SCL_BAND)
        cube = cube.mask((scl == 3) | (scl == 8) | (scl == 9) | (scl == 10))
        cube = cube.filter_bands(LR_BANDS)
        cube = cube.resample_spatial(
            resolution=10, method="bilinear",
            projection=int(S2_TARGET_CRS.split(":")[1]),
        )
        cube = cube.aggregate_temporal_period(period="month", reducer="median")
        job = cube.execute_batch(title=f"S2 monthly · rayong_{q}", out_format="GTiff")
        job.get_results().download_files(str(out))
        print(f"  done · {len(list(out.glob('openEO_*.tif')))} tifs cached")

print("\nLR cache state per quadrant:")
for q in QUADRANTS:
    have = sorted(p.name for p in lr_dir_for(q).glob("openEO_*.tif"))
    print(f"  {q}: {have}")


LR cache missing for: ['nw', 'ne']
Authenticated using refresh token.
connected: Copernicus Data Space Ecosystem openEO API

[NW] bbox={'west': 100.9795, 'south': 12.848899999999999, 'east': 101.4341, 'north': 13.168500000000002, 'crs': 'EPSG:4326'} months=['2024-10', '2024-11', '2024-12'] -> D:\Github\rayong-tracker\data\_cache\s2_monthly\rayong_nw
0:00:00 Job 'j-26051607142442a5bc1cead7ea24b0e3': send 'start'
0:00:14 Job 'j-26051607142442a5bc1cead7ea24b0e3': created (progress 0%)
0:00:20 Job 'j-26051607142442a5bc1cead7ea24b0e3': created (progress 0%)
0:00:26 Job 'j-26051607142442a5bc1cead7ea24b0e3': created (progress 0%)
0:00:35 Job 'j-26051607142442a5bc1cead7ea24b0e3': created (progress 0%)
0:00:45 Job 'j-26051607142442a5bc1cead7ea24b0e3': created (progress 0%)
0:00:57 Job 'j-26051607142442a5bc1cead7ea24b0e3': running (progress N/A)


In [ ]:
def load_sr_stack(q: str) -> xr.DataArray:
    d = sr_dir_for(q)
    tifs = sorted(d.glob("sr_*.tif"))
    if not tifs:
        raise FileNotFoundError(f"no SR tifs under {d}")
    sr_list = []
    for p in tifs:
        ym = "".join(c for c in p.stem if c.isdigit())[:6]
        if len(ym) < 6:
            continue
        t = pd.to_datetime(ym, format="%Y%m")
        sr = rxr.open_rasterio(p, masked=True, chunks={"x": 2048, "y": 2048})
        sr_list.append(sr.expand_dims(time=[t]))
    return xr.concat(sr_list, dim="time", join="exact")


def load_lr_stack(q: str) -> xr.DataArray:
    d = lr_dir_for(q)
    tifs = sorted(d.glob("openEO_*.tif"))
    keep = [p for p in tifs if any(m in p.stem for m in LR_MONTHS)]
    if not keep:
        raise FileNotFoundError(f"no LR tifs under {d} matching {LR_MONTHS}")
    lr_list = []
    for p in keep:
        ym = p.stem.split("openEO_")[-1][:7]
        t = pd.to_datetime(ym, format="%Y-%m")
        lr = rxr.open_rasterio(p, masked=True, chunks={"x": 2048, "y": 2048})
        lr_list.append(lr.expand_dims(time=[t]))
    return xr.concat(lr_list, dim="time", join="exact").sortby("time")


In [ ]:
def rasterize_for(R: xr.DataArray):
    lu = lu_all.to_crs(R.rio.crs).copy()
    codes_in_use = sorted(lu[lu["_all_minority"]]["LU_CODE"].dropna().unique())
    code_to_int = {c: i + 1 for i, c in enumerate(codes_in_use)}
    int_to_code = {v: k for k, v in code_to_int.items()}
    lu["_int"] = lu["LU_CODE"].map(code_to_int).fillna(0).astype("int32")
    out_shape = R.isel(time=0).shape[1:]
    transform = R.rio.transform()
    shapes = [(g, v) for g, v in zip(lu.geometry, lu["_int"]) if v > 0]
    LABELS = rasterize(shapes, out_shape=out_shape, transform=transform, fill=0, dtype="int32")
    return LABELS, transform, codes_in_use, code_to_int, int_to_code, lu


## 3 · Per-pixel features · band stats + vegetation indices

Twelve-dimensional schema on each grid: `{B02,B03,B04,B08} × {mean,std}` across the three months + `NDVI_mean`, `NDVI_amp`, `NDWI_mean`, `EVI_mean`.


In [ ]:
_BAND_NAMES = ("B02", "B03", "B04", "B08")
_eps = 1e-6


def _sample_features(R: xr.DataArray, LABELS: np.ndarray, stride: int, i2c: dict) -> pd.DataFrame:
    rng = np.random.RandomState(0)
    H, W = LABELS.shape
    yy, xx = np.meshgrid(np.arange(0, H, stride), np.arange(0, W, stride), indexing="ij")
    yy, xx = yy.ravel(), xx.ravel()
    lbl = LABELS[yy, xx]
    keep = lbl > 0
    yy, xx, lbl = yy[keep], xx[keep], lbl[keep]
    keep_mask = np.zeros(len(yy), dtype=bool)
    for cls_int in np.unique(lbl):
        idx = np.where(lbl == cls_int)[0]
        if len(idx) > MAX_PIX_PER_CLASS:
            idx = rng.choice(idx, size=MAX_PIX_PER_CLASS, replace=False)
        keep_mask[idx] = True
    yy, xx, lbl = yy[keep_mask], xx[keep_mask], lbl[keep_mask]

    n_t = len(R.time)
    spec = np.zeros((len(yy), n_t, 4), dtype="float32")
    for ti in range(n_t):
        arr = R.isel(time=ti).values
        if np.nanmax(arr) > 5:
            arr = arr / 10000.0
        spec[:, ti, :] = arr[:, yy, xx].T
        del arr

    s = {}
    for b, nm in enumerate(_BAND_NAMES):
        s[f"{nm}_mean"] = spec[:, :, b].mean(axis=1)
        s[f"{nm}_std"]  = spec[:, :, b].std(axis=1)
    b02 = spec[:, :, 0]; b03 = spec[:, :, 1]; b04 = spec[:, :, 2]; b08 = spec[:, :, 3]
    ndvi = (b08 - b04) / (b08 + b04 + _eps)
    ndwi = (b03 - b08) / (b03 + b08 + _eps)
    evi  = 2.5 * (b08 - b04) / (b08 + 6.0 * b04 - 7.5 * b02 + 1.0 + _eps)
    s["NDVI_mean"] = ndvi.mean(axis=1)
    s["NDVI_amp"]  = ndvi.max(axis=1) - ndvi.min(axis=1)
    s["NDWI_mean"] = ndwi.mean(axis=1)
    s["EVI_mean"]  = evi.mean(axis=1)

    df = pd.DataFrame(s)
    df["label_int"] = lbl
    df["lu_code"]   = [i2c[int(i)] for i in lbl]
    df["is_mixed"]  = df["lu_code"].apply(lambda c: "/" in c)
    df["primary_minority"] = df["lu_code"].apply(
        lambda c: next((MINORITY_TAXONOMY[x] for x in _components(c) if x in MINORITY_TAXONOMY), "?")
    )
    return df


SR_BY_Q: dict = {}; LR_BY_Q: dict = {}
LABELS_SR_BY_Q: dict = {}; LABELS_LR_BY_Q: dict = {}
TRANSFORM_SR_BY_Q: dict = {}; TRANSFORM_LR_BY_Q: dict = {}
LU_SR_BY_Q: dict = {}; LU_LR_BY_Q: dict = {}
PIX_SR_FRAMES: list = []; PIX_LR_FRAMES: list = []

for q in QUADRANTS:
    print(f"\n=== {q.upper()} ===")
    SR_q = load_sr_stack(q)
    LR_q = load_lr_stack(q)
    SR_BY_Q[q] = SR_q; LR_BY_Q[q] = LR_q

    LABELS_sr, T_sr, _, _, i2c_sr, lu_sr = rasterize_for(SR_q)
    LABELS_lr, T_lr, _, _, i2c_lr, lu_lr = rasterize_for(LR_q)
    LABELS_SR_BY_Q[q] = LABELS_sr; LABELS_LR_BY_Q[q] = LABELS_lr
    TRANSFORM_SR_BY_Q[q] = T_sr;   TRANSFORM_LR_BY_Q[q] = T_lr
    LU_SR_BY_Q[q] = lu_sr;         LU_LR_BY_Q[q] = lu_lr
    print(f"  SR {SR_q.shape}  LR {LR_q.shape}")

    df_sr = _sample_features(SR_q, LABELS_sr, STRIDE_SR, i2c_sr); df_sr["quadrant"] = q; df_sr["grid"] = "SR"
    df_lr = _sample_features(LR_q, LABELS_lr, STRIDE_LR, i2c_lr); df_lr["quadrant"] = q; df_lr["grid"] = "LR"
    print(f"  SR sampled {len(df_sr):,}  LR sampled {len(df_lr):,}")
    PIX_SR_FRAMES.append(df_sr); PIX_LR_FRAMES.append(df_lr)

PIX_SR = pd.concat(PIX_SR_FRAMES, ignore_index=True)
PIX_LR = pd.concat(PIX_LR_FRAMES, ignore_index=True)
PIX = PIX_SR
print(f"\ncombined PIX_SR: {PIX_SR.shape}")
print(f"combined PIX_LR: {PIX_LR.shape}")
print(f"\nSR per-quadrant rows:\n{PIX_SR['quadrant'].value_counts().to_string()}")
print(f"\nLR per-quadrant rows:\n{PIX_LR['quadrant'].value_counts().to_string()}")


## 4 · KMeans on pure-minority pixels · LR vs SR

Sanity check before any mixed-polygon decomposition. If KMeans cannot separate the pure components on either grid, no per-pair test can succeed.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

FEATURE_COLS = [c for c in PIX_SR.columns
                if c not in ("label_int", "lu_code", "is_mixed", "primary_minority", "quadrant", "grid")]


def fit_pure(df: pd.DataFrame) -> tuple[pd.DataFrame, StandardScaler, np.ndarray, KMeans, dict]:
    pure = df[~df["is_mixed"]].copy()
    classes = sorted(pure["primary_minority"].unique())
    K = len(classes)
    scaler = StandardScaler().fit(pure[FEATURE_COLS].values)
    X = scaler.transform(pure[FEATURE_COLS].values)
    km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X)
    pure["cluster"] = km.labels_
    centroids = {cn: X[pure["primary_minority"].values == cn].mean(0) for cn in classes}
    return pure, scaler, X, km, centroids


pure_SR, scaler_SR, X_pure_SR, km_SR, centroids_SR = fit_pure(PIX_SR)
pure_LR, scaler_LR, X_pure_LR, km_LR, centroids_LR = fit_pure(PIX_LR)

def _report(name, pure, km):
    K = pure["primary_minority"].nunique()
    ari = adjusted_rand_score(pure["primary_minority"], pure["cluster"])
    nmi = normalized_mutual_info_score(pure["primary_minority"], pure["cluster"])
    print(f"{name:<3s}  N={len(pure):>9,}  K={K}  ARI={ari:+.3f}  NMI={nmi:.3f}")

print("pure-minority KMeans · LR vs SR")
_report("SR", pure_SR, km_SR)
_report("LR", pure_LR, km_LR)

print("\nLR · cluster × class contingency:")
print(pd.crosstab(pure_LR["cluster"], pure_LR["primary_minority"]).to_string())
print("\nSR · cluster × class contingency:")
print(pd.crosstab(pure_SR["cluster"], pure_SR["primary_minority"]).to_string())


### 4.1 · Pairwise Fisher separability

Pairwise Fisher distance between pure-component centroids in standardised feature space. Higher = more separable; the matrix is the upper bound on what any clustering on these features can achieve.


In [ ]:
def fisher_matrix(X: np.ndarray, labels: np.ndarray, classes: list[str]) -> np.ndarray:
    F = np.zeros((len(classes), len(classes)))
    by = {cn: X[labels == cn] for cn in classes}
    for i, a in enumerate(classes):
        Xa = by[a]
        for j in range(i + 1, len(classes)):
            b = classes[j]; Xb = by[b]
            mu = Xa.mean(0) - Xb.mean(0)
            var = 0.5 * (Xa.var(0) + Xb.var(0)) + 1e-6
            F[i, j] = F[j, i] = float(np.sqrt((mu ** 2 / var).sum()))
    return F

classes = sorted(pure_SR["primary_minority"].unique())
F_SR = fisher_matrix(X_pure_SR, pure_SR["primary_minority"].values, classes)
F_LR = fisher_matrix(X_pure_LR, pure_LR["primary_minority"].values, classes)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, F, label in zip(axes, [F_LR, F_SR], ["LR (10 m)", "SR (2.5 m)"]):
    im = ax.imshow(F, cmap="viridis", vmin=0, vmax=max(F_LR.max(), F_SR.max()))
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{F[i,j]:.1f}", ha="center", va="center", color="w", fontsize=7)
    ax.set_title(f"Fisher distance · {label}")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

deltas = []
for i, a in enumerate(classes):
    for j in range(i + 1, len(classes)):
        b = classes[j]
        deltas.append({"A": a, "B": b, "fisher_LR": round(float(F_LR[i, j]), 3),
                       "fisher_SR": round(float(F_SR[i, j]), 3),
                       "delta_SR_minus_LR": round(float(F_SR[i, j] - F_LR[i, j]), 3)})
SEP_DELTA = pd.DataFrame(deltas).sort_values("delta_SR_minus_LR", ascending=False).reset_index(drop=True)
print("Fisher-distance delta (SR - LR), sorted descending (positive = SR more separable):")
print(SEP_DELTA.to_string(index=False))


## 5 · UMAP · pure-minority embedding

Project the 12-D feature space to 2-D. Distinct blobs per class = the schema carries class-specific signal.


In [ ]:
try:
    import umap
    HAVE_UMAP = True
except Exception:
    HAVE_UMAP = False
    print("umap-learn not installed")

if HAVE_UMAP:
    SUB = 15_000
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    for ax, X, pure, label in zip(axes, [X_pure_LR, X_pure_SR], [pure_LR, pure_SR], ["LR (10 m)", "SR (2.5 m)"]):
        n = min(SUB, len(pure))
        idx = np.random.RandomState(0).choice(len(pure), size=n, replace=False)
        emb = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0).fit_transform(X[idx])
        cls = pure["primary_minority"].values[idx]
        for c in classes:
            m = cls == c
            ax.scatter(emb[m, 0], emb[m, 1], s=3, alpha=0.5, label=c)
        ax.set_title(f"UMAP · pure pixels · {label}")
        ax.legend(loc="best", fontsize=7, markerscale=2)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()


### 5.1 · Vegetation-index distributions per pure class

Violin plots of `NDVI_mean`, `NDVI_amp`, `NDWI_mean`, `EVI_mean` per pure minority class. Diagnostic for class-specific phenological signal in the feature schema.


In [ ]:
viz_feats = ["NDVI_mean", "NDVI_amp", "NDWI_mean", "EVI_mean"]
fig, axes = plt.subplots(2, len(viz_feats), figsize=(5 * len(viz_feats), 9), sharex=True)
for row, (pure, label) in enumerate([(pure_LR, "LR"), (pure_SR, "SR")]):
    for ax, feat in zip(axes[row], viz_feats):
        data = [pure.loc[pure["primary_minority"] == c, feat].values for c in classes]
        parts = ax.violinplot(data, showmedians=True, widths=0.85)
        for body, color in zip(parts["bodies"], plt.cm.tab10(np.arange(len(classes)) % 10)):
            body.set_facecolor(color); body.set_alpha(0.65)
        ax.set_xticks(range(1, len(classes) + 1))
        ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
        ax.set_title(f"{feat} · {label}"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 6 · Mixed-polygon decomposition

For every minority-only mixed `LU_CODE = A/B` with sufficient pixels on both grids, run K=2 KMeans on the pool `(pure A + pure B + mixed A/B)`. Report per-component purity (how often KMeans recovers the pure label), mixed-pixel split ratio, and overlay a UMAP embedding.


In [ ]:
# Centroid-distance baseline on SR · per mixed LU_CODE, the closest pure centroid.
mixed = PIX_SR[PIX_SR["is_mixed"]].copy()
X_mixed = scaler_SR.transform(mixed[FEATURE_COLS].values)
names = list(centroids_SR)
C = np.stack([centroids_SR[n] for n in names], axis=0)
dist = np.linalg.norm(X_mixed[:, None, :] - C[None, :, :], axis=2)
nearest = dist.argmin(axis=1)
mixed["nearest_pure"] = [names[i] for i in nearest]
mixed["nearest_dist"] = dist.min(axis=1)

for code, sub in mixed.groupby("lu_code"):
    if len(sub) < 50:
        continue
    comp_names = [MINORITY_TAXONOMY[c] for c in _components(code)]
    counts = sub["nearest_pure"].value_counts(normalize=True)
    print(f"  {code:<12s} ({' + '.join(comp_names)})  N={len(sub):>5d}")
    for k, v in counts.head(3).items():
        marker = "✓" if k in comp_names else " "
        print(f"      {marker} -> {k:<12s} {v:>5.1%}")


### 6.1 · Mixed-code → pure-centroid distance

Mean Euclidean distance from every mixed-`LU_CODE` pixel cloud to each pure-class centroid (standardised SR feature space). Asterisks mark the polygon's listed components. The minimum-distance column should land on a listed component if the polygon is genuinely two-component on this AOI.


In [ ]:
def dist_heatmap(df: pd.DataFrame, scaler, centroids: dict, title: str, ax):
    mixed_codes = sorted(c for c in df["lu_code"].unique() if "/" in c)
    classes_local = sorted(centroids.keys())
    H = np.full((len(mixed_codes), len(classes_local)), np.nan)
    for i, mc in enumerate(mixed_codes):
        Xm = scaler.transform(df.loc[df["lu_code"] == mc, FEATURE_COLS].values)
        if len(Xm) == 0:
            continue
        for j, cn in enumerate(classes_local):
            H[i, j] = float(np.linalg.norm(Xm - centroids[cn], axis=1).mean())
    row_labels = [f"{mc} ({'+'.join(MINORITY_TAXONOMY[c] for c in _components(mc))})" for mc in mixed_codes]
    im = ax.imshow(H, cmap="viridis_r", aspect="auto")
    ax.set_xticks(range(len(classes_local))); ax.set_xticklabels(classes_local, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(mixed_codes))); ax.set_yticklabels(row_labels, fontsize=7)
    for i, mc in enumerate(mixed_codes):
        comp = [MINORITY_TAXONOMY[c] for c in _components(mc)]
        row_min = np.nanmin(H[i])
        for j, cn in enumerate(classes_local):
            star = "*" if cn in comp else ""
            weight = "bold" if abs(H[i, j] - row_min) < 1e-6 else "normal"
            ax.text(j, i, f"{H[i,j]:.1f}{star}", ha="center", va="center", fontsize=6, color="w", fontweight=weight)
    ax.set_title(title)
    return im

fig, axes = plt.subplots(1, 2, figsize=(15, max(4.5, 0.32 * len(set(c for c in PIX_SR['lu_code'] if '/' in c)))))
im_lr = dist_heatmap(PIX_LR, scaler_LR, centroids_LR, "LR (10 m)", axes[0])
im_sr = dist_heatmap(PIX_SR, scaler_SR, centroids_SR, "SR (2.5 m)", axes[1])
plt.colorbar(im_sr, ax=axes[1], fraction=0.04)
plt.suptitle("Mixed-code pixels → pure-centroid mean distance  (* = listed component, bold = row min)")
plt.tight_layout(); plt.show()


### 6.2 · Per-pair K=2 KMeans · LR vs SR

K=2 KMeans on the pool of pure A + pure B + mixed A/B for every eligible pair, run independently on the LR (10 m) and SR (2.5 m) feature tables. Cluster IDs are aligned to A/B by majority vote on the pure pixels.


In [ ]:
# Per-pair K=2 KMeans on (pure A + pure B + mixed A/B). LR vs SR side-by-side.
from sklearn.cluster import KMeans

MIN_PURE_PER_COMP = 200
MIN_MIXED_PIXELS  = 100
PAIR_PURE_CAP     = 10_000


def _eligible(df: pd.DataFrame, ca: str, cb: str, mc: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame] | None:
    pa = df[(~df["is_mixed"]) & (df["lu_code"] == ca)]
    pb = df[(~df["is_mixed"]) & (df["lu_code"] == cb)]
    mxn = df[df["lu_code"] == mc]
    if len(pa) < MIN_PURE_PER_COMP or len(pb) < MIN_PURE_PER_COMP or len(mxn) < MIN_MIXED_PIXELS:
        return None
    return pa, pb, mxn


def _take(df: pd.DataFrame, n: int, seed: int = 7) -> pd.DataFrame:
    if len(df) <= n:
        return df
    rng = np.random.RandomState(seed)
    return df.iloc[rng.choice(len(df), size=n, replace=False)]


def run_pair_km(df: pd.DataFrame, scaler, ca: str, cb: str, mc: str) -> dict | None:
    elig = _eligible(df, ca, cb, mc)
    if elig is None:
        return None
    pa, pb, mxn = elig
    pa_s = _take(pa, PAIR_PURE_CAP); pb_s = _take(pb, PAIR_PURE_CAP)
    nm_a = MINORITY_TAXONOMY[ca]; nm_b = MINORITY_TAXONOMY[cb]

    pool = pd.concat([
        pa_s.assign(_origin=f"pure {nm_a}"),
        pb_s.assign(_origin=f"pure {nm_b}"),
        mxn.assign(_origin=f"mixed {mc}"),
    ], ignore_index=True)
    X = scaler.transform(pool[FEATURE_COLS].values)
    km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(X)
    pool["cluster"] = km.labels_

    a_mask = pool["_origin"].str.startswith(f"pure {nm_a}")
    b_mask = pool["_origin"].str.startswith(f"pure {nm_b}")
    c_for_a = int(pool.loc[a_mask, "cluster"].mode().iloc[0])
    c_for_b = 1 - c_for_a
    pool["km_pred"] = pool["cluster"].map({c_for_a: nm_a, c_for_b: nm_b})

    pure_A_purity = (pool.loc[a_mask, "km_pred"] == nm_a).mean()
    pure_B_purity = (pool.loc[b_mask, "km_pred"] == nm_b).mean()
    mxp = pool[pool["_origin"].str.startswith("mixed")]
    mix_split_a = (mxp["km_pred"] == nm_a).mean()
    mix_split_b = (mxp["km_pred"] == nm_b).mean()

    return {
        "n_pure_A": len(pa), "n_pure_B": len(pb), "n_mixed": len(mxn),
        "pure_A_purity": round(float(pure_A_purity) * 100, 1),
        "pure_B_purity": round(float(pure_B_purity) * 100, 1),
        "balanced_purity": round(float(0.5 * (pure_A_purity + pure_B_purity)) * 100, 1),
        "mix_split_A": round(float(mix_split_a) * 100, 1),
        "mix_split_B": round(float(mix_split_b) * 100, 1),
        "pool": pool, "X": X,
    }


mixed_codes = sorted(set(PIX_SR.loc[PIX_SR["is_mixed"], "lu_code"]).union(
                     PIX_LR.loc[PIX_LR["is_mixed"], "lu_code"]))

rows: list = []
PAIR_RUNS: dict = {}
for mc in mixed_codes:
    ca, cb = mc.split("/")
    if ca not in MINORITY_TAXONOMY or cb not in MINORITY_TAXONOMY:
        continue
    res_sr = run_pair_km(PIX_SR, scaler_SR, ca, cb, mc)
    res_lr = run_pair_km(PIX_LR, scaler_LR, ca, cb, mc)
    nm_a, nm_b = MINORITY_TAXONOMY[ca], MINORITY_TAXONOMY[cb]
    rows.append({
        "pair": mc, "components": f"{nm_a} + {nm_b}",
        "n_mixed_SR": (res_sr or {}).get("n_mixed"),
        "n_mixed_LR": (res_lr or {}).get("n_mixed"),
        "LR_balanced_purity":  (res_lr or {}).get("balanced_purity"),
        "SR_balanced_purity":  (res_sr or {}).get("balanced_purity"),
        "delta_SR_minus_LR":   None if (not res_sr or not res_lr) else round(res_sr["balanced_purity"] - res_lr["balanced_purity"], 1),
        "LR_mix_split_A":  (res_lr or {}).get("mix_split_A"),
        "SR_mix_split_A":  (res_sr or {}).get("mix_split_A"),
    })
    PAIR_RUNS[mc] = {"SR": res_sr, "LR": res_lr}

PAIR_KM = pd.DataFrame(rows).sort_values("delta_SR_minus_LR", ascending=False, na_position="last").reset_index(drop=True)
print("Per-pair K=2 KMeans · balanced pure-component purity, LR vs SR")
print("(delta > 0 = SR clusters separate the pair more cleanly than LR)\n")
print(PAIR_KM.to_string(index=False))


### 6.3 · Per-pair UMAP · LR vs SR

UMAP scatter per pair, side-by-side LR and SR. Points coloured by the KMeans cluster assignment. Two distinct blobs that match the pure-component clouds = the grid separates the components; one blob = it does not.


In [ ]:
# Per-pair UMAP scatter, LR vs SR side-by-side, points coloured by KMeans cluster.
if HAVE_UMAP:
    eligible_pairs = [mc for mc, runs in PAIR_RUNS.items() if runs["SR"] is not None and runs["LR"] is not None]
    eligible_pairs = sorted(eligible_pairs, key=lambda mc: -(PAIR_RUNS[mc]["SR"]["n_mixed"] or 0))[:8]

    rows = len(eligible_pairs)
    fig, axes = plt.subplots(rows, 2, figsize=(11, 4 * rows), squeeze=False)
    palette = {"A": "#3F7D58", "B": "#C96442", "mix": "#7B5BA6"}

    for r, mc in enumerate(eligible_pairs):
        ca, cb = mc.split("/")
        nm_a, nm_b = MINORITY_TAXONOMY[ca], MINORITY_TAXONOMY[cb]
        for col, grid in enumerate(["LR", "SR"]):
            res = PAIR_RUNS[mc][grid]
            ax = axes[r, col]
            pool = res["pool"]
            emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(res["X"])
            # Origin markers (shape) × cluster colour.
            markers = {f"pure {nm_a}": "o", f"pure {nm_b}": "s", f"mixed {mc}": "^"}
            for origin, mk in markers.items():
                m = pool["_origin"].values == origin
                if not m.any():
                    continue
                colours = np.where(pool.loc[m, "km_pred"].values == nm_a, palette["A"], palette["B"])
                ax.scatter(emb[m, 0], emb[m, 1], s=8, alpha=0.55, marker=mk, c=colours, label=origin)
            ax.set_title(f"{mc} · {nm_a}+{nm_b} · {grid}  (purity {res['balanced_purity']:.1f}%)", fontsize=9)
            ax.legend(fontsize=6, markerscale=1.5, loc="best")
            ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()


### 6.4 · Gaussian-mixture soft posterior · SR

Two-component GMM initialised at the pure-class centroids. Reports the posterior probability per mixed pixel, which can weight downstream training.


In [ ]:
from sklearn.mixture import GaussianMixture

GMM_TARGET = "A403/A419"
ca, cb = GMM_TARGET.split("/")
nm_a, nm_b = MINORITY_TAXONOMY[ca], MINORITY_TAXONOMY[cb]

pa = PIX_SR[(~PIX_SR["is_mixed"]) & (PIX_SR["lu_code"] == ca)]
pb = PIX_SR[(~PIX_SR["is_mixed"]) & (PIX_SR["lu_code"] == cb)]
mxn = PIX_SR[PIX_SR["lu_code"] == GMM_TARGET]
print(f"{GMM_TARGET}: pure {nm_a}={len(pa):,}  pure {nm_b}={len(pb):,}  mixed={len(mxn):,}")
if len(pa) == 0 or len(pb) == 0 or len(mxn) == 0:
    raise RuntimeError("missing component pixels in SR for GMM_TARGET")

rng = np.random.RandomState(0)
def _take_g(df, n):
    if len(df) <= n: return df
    return df.iloc[rng.choice(len(df), size=n, replace=False)]
pa_s = _take_g(pa, 5_000); pb_s = _take_g(pb, 5_000); mx_s = _take_g(mxn, 5_000)

Xa = scaler_SR.transform(pa_s[FEATURE_COLS].values)
Xb = scaler_SR.transform(pb_s[FEATURE_COLS].values)
Xm = scaler_SR.transform(mxn[FEATURE_COLS].values)
mu_init = np.stack([Xa.mean(0), Xb.mean(0)], axis=0)
X_fit = np.concatenate([Xa, Xb, scaler_SR.transform(mx_s[FEATURE_COLS].values)], axis=0)
gmm = GaussianMixture(n_components=2, means_init=mu_init, covariance_type="full",
                       random_state=0, max_iter=200).fit(X_fit)

which_is_a = int(np.argmin(np.linalg.norm(gmm.means_ - Xa.mean(0), axis=1)))
which_is_b = 1 - which_is_a
post = gmm.predict_proba(Xm)
p_a = post[:, which_is_a]; p_b = post[:, which_is_b]
print(f"\nmean P({nm_a})={p_a.mean():.2f}  P({nm_b})={p_b.mean():.2f}")
print(f"high-confidence {nm_a} (P>0.8): {(p_a > 0.8).sum():,}  ({(p_a > 0.8).mean():.1%})")
print(f"high-confidence {nm_b} (P>0.8): {(p_b > 0.8).sum():,}  ({(p_b > 0.8).mean():.1%})")
print(f"ambiguous (0.4<P<0.6):          {((p_a > 0.4) & (p_a < 0.6)).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(p_a, bins=40, edgecolor="k", color="#3F7D58", alpha=0.85)
axes[0].axvline(0.5, color="red", lw=1, ls="--")
axes[0].set_xlabel(f"P({nm_a})"); axes[0].set_ylabel("mixed-pixel count")
axes[0].set_title(f"GMM posterior · mixed {GMM_TARGET}")
axes[1].hist(gmm.predict_proba(Xa)[:, which_is_a], bins=40, alpha=0.6, color="#3F7D58", edgecolor="k", label=f"pure {nm_a}")
axes[1].hist(gmm.predict_proba(Xb)[:, which_is_a], bins=40, alpha=0.6, color="#C96442", edgecolor="k", label=f"pure {nm_b}")
axes[1].axvline(0.5, color="red", lw=1, ls="--")
axes[1].set_xlabel(f"P({nm_a})"); axes[1].set_ylabel("pure-pixel count")
axes[1].set_title("GMM sanity · pure components"); axes[1].legend()
plt.tight_layout(); plt.show()


### 6.5 · Spatial render · KMeans assignment on one mixed polygon

Pick the largest mixed polygon of a target pair; render the SR-grid per-pixel KMeans cluster assignment back on the polygon footprint. Spatially coherent A- and B-regions = the decomposition is recovering real structure; salt-and-pepper = noise.


In [ ]:
from rasterio.windows import Window
from rasterio.windows import transform as _win_transform
from rasterio.transform import rowcol

SPATIAL_TARGET = "A403/A419"
SPATIAL_TARGET_Q = None

ca, cb = SPATIAL_TARGET.split("/")
nm_a, nm_b = MINORITY_TAXONOMY[ca], MINORITY_TAXONOMY[cb]

candidate_rows = []
for _q, _lu in LU_SR_BY_Q.items():
    if SPATIAL_TARGET_Q is not None and _q != SPATIAL_TARGET_Q:
        continue
    sub = _lu[_lu["LU_CODE"] == SPATIAL_TARGET].copy()
    if len(sub) == 0:
        continue
    sub["_area"] = sub.geometry.area
    sub["_q"] = _q
    candidate_rows.append(sub)
if not candidate_rows:
    raise RuntimeError(f"no {SPATIAL_TARGET} polygons in any quadrant {list(LU_SR_BY_Q)}")

cands = pd.concat(candidate_rows, ignore_index=True)
poly = cands.sort_values("_area", ascending=False).iloc[0]
poly_q = poly["_q"]
SR_poly = SR_BY_Q[poly_q]
LABELS_poly = LABELS_SR_BY_Q[poly_q]
transform_poly = TRANSFORM_SR_BY_Q[poly_q]
print(f"{SPATIAL_TARGET}: chose polygon in {poly_q.upper()} area={poly['_area']:.0f} m²")

minx, miny, maxx, maxy = poly.geometry.bounds
r0, c0 = rowcol(transform_poly, minx, maxy)
r1, c1 = rowcol(transform_poly, maxx, miny)
r0, r1 = sorted((max(0, r0), min(LABELS_poly.shape[0], r1)))
c0, c1 = sorted((max(0, c0), min(LABELS_poly.shape[1], c1)))
H_p, W_p = r1 - r0, c1 - c0
print(f"polygon SR window: {H_p}×{W_p} px ({H_p * W_p:,} pixels)")

window = Window(c0, r0, W_p, H_p)
win_xform = _win_transform(window, transform_poly)
mask_poly = rasterize([(poly.geometry, 1)], out_shape=(H_p, W_p), transform=win_xform, fill=0, dtype="uint8").astype(bool)

n_t = len(SR_poly.time)
spec_p = np.zeros((n_t, 4, H_p, W_p), dtype="float32")
for ti in range(n_t):
    arr = SR_poly.isel(time=ti, y=slice(r0, r1), x=slice(c0, c1)).values
    if np.nanmax(arr) > 5:
        arr = arr / 10000.0
    spec_p[ti] = arr

feats = {}
for b, nm in enumerate(_BAND_NAMES):
    feats[f"{nm}_mean"] = spec_p[:, b].mean(0)
    feats[f"{nm}_std"]  = spec_p[:, b].std(0)
b02, b03, b04, b08 = spec_p[:, 0], spec_p[:, 1], spec_p[:, 2], spec_p[:, 3]
ndvi = (b08 - b04) / (b08 + b04 + _eps)
ndwi = (b03 - b08) / (b03 + b08 + _eps)
evi  = 2.5 * (b08 - b04) / (b08 + 6.0 * b04 - 7.5 * b02 + 1.0 + _eps)
feats["NDVI_mean"] = ndvi.mean(0); feats["NDVI_amp"] = ndvi.max(0) - ndvi.min(0)
feats["NDWI_mean"] = ndwi.mean(0); feats["EVI_mean"]  = evi.mean(0)

X_poly = np.stack([feats[c] for c in FEATURE_COLS], axis=-1).reshape(-1, len(FEATURE_COLS))
X_poly = scaler_SR.transform(np.nan_to_num(X_poly, nan=0.0))

# Per-pair K=2 fit on (pure A + pure B + mixed A/B) from PIX_SR; assign cluster
# IDs to A / B by majority on pure pixels; apply to polygon pixels.
res = run_pair_km(PIX_SR, scaler_SR, ca, cb, SPATIAL_TARGET)
if res is None:
    raise RuntimeError(f"{SPATIAL_TARGET} not eligible in PIX_SR; widen MIN_PURE_PER_COMP")
km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(res["X"])
pure_centroid_A = res["X"][res["pool"]["_origin"].str.startswith(f"pure {nm_a}")].mean(0)
mu_diff = km.cluster_centers_ - pure_centroid_A
c_for_a = int(np.argmin(np.linalg.norm(mu_diff, axis=1)))
c_for_b = 1 - c_for_a

poly_cluster = km.predict(X_poly)
label_img = np.where(poly_cluster == c_for_a, 0, 1).reshape(H_p, W_p).astype(np.int8)
label_img[~mask_poly] = -1

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
rgb = np.stack([
    np.clip(spec_p[0, 2] * 3, 0, 1),
    np.clip(spec_p[0, 1] * 3, 0, 1),
    np.clip(spec_p[0, 0] * 3, 0, 1),
], axis=-1)
axes[0].imshow(rgb); axes[0].set_title(f"True colour · month {str(SR_poly.time.values[0])[:7]}")
axes[0].set_xticks([]); axes[0].set_yticks([])

overlay = np.zeros((H_p, W_p, 3), dtype="float32")
overlay[label_img == 0] = (0.247, 0.490, 0.345)
overlay[label_img == 1] = (0.788, 0.392, 0.259)
overlay[label_img == -1] = (0.1, 0.1, 0.1)
axes[1].imshow(overlay)
axes[1].set_title(f"KMeans cluster · A={nm_a} (green) · B={nm_b} (orange)")
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()


## 7 · Export decomposed mixed pixels

Persist mixed pixels labelled by per-pair KMeans (cluster assignment + posterior distance) for the downstream RF training pool.


In [ ]:
# Persist all eligible mixed pixels labelled by per-pair KMeans on the SR grid.
frames = []
for mc, runs in PAIR_RUNS.items():
    res = runs["SR"]
    if res is None:
        continue
    ca, cb = mc.split("/")
    nm_a, nm_b = MINORITY_TAXONOMY[ca], MINORITY_TAXONOMY[cb]
    pool = res["pool"]
    mxp = pool[pool["_origin"].str.startswith("mixed")].copy()
    mxp["pair"] = mc
    mxp["component_A"] = nm_a
    mxp["component_B"] = nm_b
    mxp["inferred_class"] = mxp["km_pred"]
    frames.append(mxp)

if not frames:
    print("no eligible mixed pixels to persist")
else:
    OUT = pd.concat(frames, ignore_index=True)
    out_cols = FEATURE_COLS + ["label_int", "lu_code", "quadrant", "pair", "component_A", "component_B", "inferred_class"]
    OUT = OUT[[c for c in out_cols if c in OUT.columns]]
    out_path = OUT_DIR / f"rayong_{AOI_TAG}_mixed_kmeans.parquet"
    try:
        OUT.to_parquet(out_path)
    except Exception:
        out_path = out_path.with_suffix(".pkl")
        OUT.to_pickle(out_path)
    print(f"wrote {len(OUT):,} rows -> {out_path}")
    print(OUT["inferred_class"].value_counts().to_string())
